<a href="https://colab.research.google.com/github/Harshavardhan-123-AU-CSE/Breast-Cancer-Classification-/blob/main/SkillMap_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [43]:
!pip install langchain

In [44]:
!pip install -U langchain-google-genai

In [45]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from google.colab import userdata


In [46]:
gemini_api_key = userdata.get("Gemini_api_key")
model = init_chat_model(
   "google_genai:gemini-2.5-flash",
   api_key = gemini_api_key
)

In [47]:
!pip install langchain-tavily

In [48]:
from langchain_tavily import TavilySearch


In [49]:
tavily_api_key = userdata.get("Tavily_api_key")
skill_demand = TavilySearch(
    max_results = 5,
    search_depth = "advanced",
    tavily_api_key = tavily_api_key
)
result = skill_demand.invoke({"query": "generative ai skills demand 2025"})
print(result)

{'query': 'generative ai skills demand 2025', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://aws.amazon.com/executive-insights/content/top-generative-ai-skills-and-education-trends-for-2025/', 'title': 'Top Generative AI Skills and Education Trends for 2025 | AWS Executive Insights', 'content': 'In fact, two out of three organizations said that they are increasing their investments in generative AI due to early signs of business value, according to a Deloitte study. The greater investment in generative AI means businesses will need more people with AI skills to execute on their AI strategies and roadmap, which in turn requires leaders to continue providing training for their employees in 2025. This underscores how AI skills are crucial for businesses seeking to stay competitive and the increasing importance of AI workforce development. We saw significant demand in 2024 from AWS’s customers and partners who want to help their employees gain AI fl

In [50]:
import requests
from langchain.tools import tool
from google.colab import userdata

@tool
def search_jobs(skill: str, location: str) -> list:
    """Search for jobs requiring a specific skill using JSearch API from RapidAPI."""
    print(f"\nCalling search_jobs tool")
    print(f"Searching jobs for: {skill} in {location}")

    rapidapi_key = userdata.get('RAPIDAPI_KEY')

    url = "https://jsearch.p.rapidapi.com/search"
    headers = {
        "x-rapidapi-key": rapidapi_key,
        "x-rapidapi-host": "jsearch.p.rapidapi.com"
    }
    querystring = {
        "query": f"{skill} in {location}",
        "page": "1",
        "country": "in",
        "employment_types": "INTERN,FULLTIME",
        "job_requirements": "no_experience,under_3_years_experience"
    }

    response = requests.get(url, headers=headers, params=querystring)
    data = response.json()

    jobs = data.get("data", [])
    print(f"Found {len(jobs)} jobs\n")

    result = []
    for job in jobs:
        result.append({
            "title": job.get("job_title"),
            "company": job.get("employer_name"),
            "location": job.get("job_city"),
            "apply_link": job.get("job_apply_link")
        })
    return result

In [51]:
system_prompt = """You are a Skill-to-Career Mapping assistant that helps students understand skill demand and find matching job opportunities.

You have access to these tools:
- skill_demand_tool: Search for industry demand, salary insights, and career trends
- search_jobs: Find actual job listings requiring specific skills

Help the student by researching the skill they ask about and finding relevant opportunities.

Present results in a clean, readable format with clear sections and proper spacing. Include all job details with apply links. Don't use markdown format."""
agent = create_agent(
    model=model,
    tools = [skill_demand,search_jobs],
    system_prompt= system_prompt
)

In [52]:

from pprint import pprint
user_query = "What's the demand for web developer in the industry and show me related job openings in India"

response = agent.invoke({
    "messages": [{"role": "user", "content": user_query}]
})
pprint(response["messages"][-1].content)


Calling search_jobs tool
Searching jobs for: web developer in India
Found 0 jobs

[{'extras': {'signature': 'CoEKAQw51sehed1WKPb8XU/WKuCZyatpGn2OOYhd15541rESH61p3kaTcTnsrQkErECUHcCj7a2Q0vUC+4lpVGU6d6Pu6UwlRo08Jr5Py1owbyGoMNfVoHs3ujFsqCC5kZvyWPtbr9gacngfmMc0k/57trS1Zi4PdAv/3EUgEhb/ZahQj2Ez+08Jbd+YZXPpsC0rRCAzEp5MkCd7Hiy7gYsAFqopk+vZIE2GcrlaJar4O5bYx1/rwl9qvK4S6qMxlxYZuwvbWnee95S3mJW6wuOog8QNsT4mH5CqTlaoaBStR9PW3g9KOYntQ8WELd2Pk+tVRFP/6pi6eYimWHp9BlgPzdQBwgtqMHFGBzAvB4s12EJbAUUexlxIXp63kWpxoJtiVFzIZAsgmC2njDHQ7opoLtumvQI2N9BPUKBrqaSdMFyHBpomlYHj5z6rzLjQk8ON2boWNQZ/UFkvTdUE9WGCRdITPP1xlBlkQV36OXEGnNLN8JBoTL6rBBuW4A0yiQFiXXGioFv9jYy8+kEOcAEZ9mdMmtxq76e11D4MnqQKBpW2L0U7zG1ErUAs9BdQzu5MZlfC/IFzDaRV1jC6TnT5++1cP+x4fZjVvQ49p0LtSIhQAamt0sBFlb83ZrqGJWEraMchpFTd0suDPraVcL1pF+GJUhQYpG32Fwk2AkhKGn8blKLq9tvKXrm+2Z7rj2m99TXy8A9bmCoD3w7223Z1A0je88TCT4yvpx+Es4c/iX1wmlh9iDKSLY9OolPk5h/dHuCFEys9sjZo/WYNaJtTnGEaN1+uQHnwFTO4nxG9eVZo/IddUYBxYU/aADcvuXqtTAwXGgkpi1LW8/+ymu0HpII3yj4Ay9CKWoHzSuJOS9kHbcD/FhxGQ3